# NYC Taxi Trip Duration — Model Comparison

Benchmark different models before Feature Engineering improvements.

In [6]:
import pandas as pd
import numpy as np
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("NYC Taxi - Model Comparison")


In [7]:
df = pd.read_csv("../data/processed_taxi.csv")
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dayofyear"] = df["pickup_datetime"].dt.dayofyear

base_categorical_cols = [
    "vendor_id",
    "store_and_fwd_flag",
    "passenger_count",
    "same_location"
]

numerical_cols = [
    "log_distance",
    "distance_km",
    "lat_diff",
    "lon_diff",
    "hour",
    "dayofweek",
    "month",
    "dayofyear"
]

coordinate_cols = [
    "pickup_latitude", "pickup_longitude",
    "dropoff_latitude", "dropoff_longitude"
]

feature_cols = base_categorical_cols + numerical_cols + coordinate_cols
X = df[feature_cols].copy()
y = df["log_trip_duration"].copy()


In [8]:
# 1) Split BEFORE fitting KMeans
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.copy()
X_val = X_val.copy()

# 2) Learn geographic centroids from TRAIN only
N_CLUSTERS = 5

pickup_kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
dropoff_kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")

pickup_kmeans.fit(X_train[["pickup_latitude", "pickup_longitude"]])
dropoff_kmeans.fit(X_train[["dropoff_latitude", "dropoff_longitude"]])

# 3) Assign train/validation rows using the SAME learned centroids
X_train["pickup_cluster"] = pickup_kmeans.predict(
    X_train[["pickup_latitude", "pickup_longitude"]]
)
X_val["pickup_cluster"] = pickup_kmeans.predict(
    X_val[["pickup_latitude", "pickup_longitude"]]
)

X_train["dropoff_cluster"] = dropoff_kmeans.predict(
    X_train[["dropoff_latitude", "dropoff_longitude"]]
)
X_val["dropoff_cluster"] = dropoff_kmeans.predict(
    X_val[["dropoff_latitude", "dropoff_longitude"]]
)

# Cluster IDs are labels, not ordered numbers -> categorical/OHE
categorical_cols = base_categorical_cols + ["pickup_cluster", "dropoff_cluster"]

# Raw coordinates were only needed to generate the cluster labels.
model_features = categorical_cols + numerical_cols
X_train_model = X_train[model_features]
X_val_model = X_val[model_features]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])


In [9]:
models = {
    "Ridge": Ridge(alpha=1),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )
}


In [10]:
results = []

for name, model in models.items():
    with mlflow.start_run(run_name=f"{name} - leakage-safe clusters"):
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipe.fit(X_train_model, y_train)
        pred = pipe.predict(X_val_model)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        r2 = r2_score(y_val, pred)

        mlflow.log_param("n_clusters", N_CLUSTERS)
        mlflow.log_param("cluster_fit_scope", "train_only")
        mlflow.log_param("uses_log_distance", True)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        results.append({
            "Model": name,
            "RMSE": rmse,
            "R2": r2
        })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
results_df


,Model,RMSE,R2
1,Random Forest,0.516074,0.594183
2,Gradient Boosting,0.518199,0.590834
3,XGBoost,0.525665,0.578959
0,Ridge,0.547371,0.543469


## What changed?

- The train/validation split happens **before** KMeans.
- Pickup and dropoff KMeans are fitted using **training coordinates only**.
- Validation cluster IDs are produced with `.predict()` using the training centroids.
- `pickup_cluster` and `dropoff_cluster` are treated as **categorical** features and One-Hot Encoded.
- Ridge remains `Ridge(alpha=1)` for the course project.
- Each model run is logged to MLflow with `cluster_fit_scope=train_only`.
